In [ ]:
# Copyright 2026 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

<table align="left">
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/vertex-ai/colab/import/https:%2F%2Fraw.githubusercontent.com%2Fgooglemaps-samples%2Finsights-samples%2Fmain%2Fstreet_view_insights%2Ffull_frame%2Fcrop_and_classify_pipeline.ipynb?utm_source=full_frame_street_view_insights_notebooks">
      <img width="32px" src="https://lh3.googleusercontent.com/JmcxdQi-qOpctIvWKgPtrzZdJJK-J3sWE1RsfjZNwshCFgE_9fULcNpuXYTilIR2hjwN" alt="Google Cloud Colab Enterprise logo"><br> Open in Colab Enterprise
    </a>
  </td>
</table>

# Crop and Classify Pipeline with Gemini 3.5 Flash

This notebook demonstrates how to resolve the limitation of classifying small assets within wide-angle Full-Frame Street View images by programmatically cropping the asset using BigQuery bounding box coordinates prior to invoking Gemini 3.5 Flash.

## Install Required Libraries

In [ ]:
!pip install --upgrade google-cloud-bigquery google-genai google-cloud-storage "pillow<11.0.0" matplotlib

## Configuration

**Important**: Replace the placeholder values below with your actual GCP Project ID and Region.

In [ ]:
PROJECT_ID = 'imagery-insights-sandbox'  # @param {type:"string"}
REGION = 'global'      # @param {type:"string"}

# BigQuery Configuration
BIGQUERY_DATASET_ID = 'imagery_insights___us' # @param {type:"string"}
BIGQUERY_TABLE_ID = 'full_frame_observations_latest' # @param {type:"string"}
ASSET_LIMIT = 2 # @param {type:"integer"}
ASSET_TYPE = "ASSET_CLASS_UTILITY_POLE" # @param {type:"string"}
MODEL = "gemini-3.5-flash" # @param {type:"string"}
THINKING_LEVEL = "HIGH" # @param ["MINIMAL", "LOW", "MEDIUM", "HIGH"] {type:"string"}

## Imports and SDK Initialization

In [ ]:
import io
import vertexai
import PIL.Image
import PIL.ImageDraw
import matplotlib.pyplot as plt
from google.cloud import bigquery
from google.cloud import storage
from google import genai
from google.genai import types
from google.genai.types import Content, Part

# Initialize Vertex AI SDK and Gemini Client
vertexai.init(project=PROJECT_ID, location=REGION)
client = genai.Client(vertexai=True, project=PROJECT_ID, location=REGION)

## Fetch Observations from BigQuery

We query the BigQuery table to get the GCS URIs of the images and their bounding box coordinates.

In [ ]:
BIGQUERY_SQL_QUERY = f"""
SELECT
  asset_id,
  gcs_uri,
  bbox,
  asset_type
FROM
  `{PROJECT_ID}.{BIGQUERY_DATASET_ID}.{BIGQUERY_TABLE_ID}`
WHERE asset_type = '{ASSET_TYPE}'
LIMIT {ASSET_LIMIT};
"""

try:
    bigquery_client = bigquery.Client(project=PROJECT_ID)
    query_job = bigquery_client.query(BIGQUERY_SQL_QUERY)
    query_response_data = [dict(row) for row in query_job]
    
    observations = []
    for item in query_response_data:
        if item.get("gcs_uri") and item.get("bbox"):
            observations.append({
                "asset_id": item.get("asset_id"),
                "gcs_uri": item.get("gcs_uri"),
                "bbox": item.get("bbox"),
                "asset_type": item.get("asset_type")
            })

    print(f"Successfully fetched {len(observations)} observations.")
    for obs in observations:
        print(obs['gcs_uri'])
except Exception as e:
    print(f"An error occurred while querying BigQuery: {e}")

## Visual Crop and Pipeline Helpers

Define helpers to download images from GCS, crop them programmatically using coordinates, and display comparisons.

In [ ]:
def download_image(gcs_uri: str) -> PIL.Image.Image:
    parts = gcs_uri[5:].split("/", 1)
    bucket_name = parts[0]
    blob_name = parts[1]
    storage_client = storage.Client(project=PROJECT_ID)
    bucket = storage_client.bucket(bucket_name)
    blob = bucket.blob(blob_name)
    image_bytes = blob.download_as_bytes()
    return PIL.Image.open(io.BytesIO(image_bytes))

def draw_bbox(image: PIL.Image.Image, bbox: dict, label: str = None) -> PIL.Image.Image:
    draw = PIL.ImageDraw.Draw(image)
    xmin = bbox['lo']['x']
    ymin = bbox['lo']['y']
    xmax = bbox['hi']['x']
    ymax = bbox['hi']['y']
    draw.rectangle([xmin, ymin, xmax, ymax], outline="red", width=10)
    if label:
        draw.text((xmin + 20, ymin + 20), label, fill="red")
    return image

def download_and_crop_image(gcs_uri: str, bbox: dict) -> PIL.Image.Image:
    """
    Downloads the full-frame image from GCS and crops it to the target bounding box coordinates.
    """
    image = download_image(gcs_uri)
    xmin = bbox['lo']['x']
    ymin = bbox['lo']['y']
    xmax = bbox['hi']['x']
    ymax = bbox['hi']['y']
    cropped_image = image.crop((xmin, ymin, xmax, ymax))
    return cropped_image

def display_pipeline_comparison(ff_uri: str, bbox: dict, cropped_img: PIL.Image.Image):
    """
    Displays the original full frame view and the programmatically cropped view side-by-side.
    """
    try:
        orig_img = download_image(ff_uri)
        orig_with_bbox = draw_bbox(orig_img.copy(), bbox, "Target Location")
        
        fig, axes = plt.subplots(1, 2, figsize=(16, 8))
        axes[0].imshow(orig_with_bbox)
        axes[0].set_title("Original Full Frame (with BBox)")
        axes[0].axis('off')
        
        axes[1].imshow(cropped_img)
        axes[1].set_title("Programmatic Crop (Input to Classifier)")
        axes[1].axis('off')
        
        plt.show()
    except Exception as e:
        print(f"Error displaying pipeline comparison: {e}")

## Define Pipeline Classification Function

This function supports classifying an image either by its GCS URI or as a PIL Image object, calculating token metrics and API cost dynamically.

In [ ]:
def classify_image_with_gemini(image_input, prompt: str) -> tuple[str, float, int, int]:
    """
    Classifies an image using Gemini. image_input can be a GCS URI string or a PIL Image object.
    Returns prediction text, calculated cost, prompt tokens, and completion tokens.
    """
    try:
        if isinstance(image_input, str):
            parts = [Part(file_data={'file_uri': image_input, 'mime_type': 'image/jpeg'})]
        else:
            parts = [image_input]
            
        contents = [prompt] + parts
        
        config = types.GenerateContentConfig(
            thinking_config=types.ThinkingConfig(
                thinking_level=THINKING_LEVEL
            )
        )
        response = client.models.generate_content(model=MODEL, contents=contents, config=config)
        
        # Calculate cost dynamically from usage metadata
        prompt_tokens = response.usage_metadata.prompt_token_count
        completion_tokens = response.usage_metadata.candidates_token_count
        
        # Pricing for gemini-3.5-flash: Input: $0.000075 / 1k, Output: $0.00030 / 1k
        input_cost = prompt_tokens * (0.000075 / 1000)
        output_cost = completion_tokens * (0.00030 / 1000)
        total_cost = input_cost + output_cost
        
        return response.text, total_cost, prompt_tokens, completion_tokens
    except Exception as e:
        print(f"Error classifying image: {e}")
        return "Classification failed.", 0.0, 0, 0

## Execute Pipeline and Compare

Loop through each observation, rendering the original and cropped views, running both classifications, and comparing the accuracy of predictions, token counts, and API costs.

In [ ]:
prompt = """You will be provided with a photo.
Analyze it and return your findings in the JSON format:
```json
{
  \"pole_condition\": \"OK/Damaged/Other Issues\",
  \"type\": \"<pole_type>\",
  \"material\": \"<material>\",
  \"transformers\": <number_of_transformers>,
  \"power_lines\": <number_of_power_lines>,
  \"street_lamps\": <number_of_street_lamps>,
  \"junction_boxes\": <number_of_junction_boxes>,
  \"additional_notes\": \"<observations>\"
}
```
"""

if 'observations' in locals() and observations:
    for obs in observations:
        uri = obs['gcs_uri']
        bbox = obs['bbox']
        aid = obs['asset_id']
        
        print(f"\n=========================================================================")
        print(f"PIPELINE RUN FOR OBSERVATION: {uri} (Asset: {aid})")
        print(f"=========================================================================")
        
        # 1. Download and crop image programmatically
        print("Downloading and programmatically cropping image...")
        cropped_img = download_and_crop_image(uri, bbox)
        
        # 2. Display side-by-side comparison
        display_pipeline_comparison(uri, bbox, cropped_img)
        
        # 3. Classify original image (using GCS URI)
        print("\n--- Classifying Original Full-Frame Image ---")
        orig_res, orig_cost, orig_in, orig_out = classify_image_with_gemini(uri, prompt)
        
        # 4. Classify cropped image (using PIL Image)
        print("\n--- Classifying Programmatically Cropped Image ---")
        crop_res, crop_cost, crop_in, crop_out = classify_image_with_gemini(cropped_img, prompt)
        
        # 5. Display Pipeline Comparison Report
        print(f"\n---------------------------------------------------------------")
        print(f"PIPELINE COMPARISON REPORT")
        print(f"---------------------------------------------------------------")
        print(f"[RAW FULL FRAME ANALYSIS]:\n{orig_res}")
        print(f"Tokens: Input: {orig_in} | Output: {orig_out} | Cost: ${orig_cost:.6f}")
        print(f"\n[CROP-THEN-CLASSIFY ANALYSIS]:\n{crop_res}")
        print(f"Tokens: Input: {crop_in} | Output: {crop_out} | Cost: ${crop_cost:.6f}")
        print(f"---------------------------------------------------------------")
else:
    print("No observations found to run the pipeline on.")